# Lezione 09 - Pattern di Progettazione Metacognitiva


## Configurazione

Questo notebook dimostra il design pattern Metacognition utilizzando il Microsoft Agent Framework.

**Prerequisiti:**
- Distribuzione Azure OpenAI configurata tramite variabili d'ambiente
- Azure CLI autenticato (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Cos'è la Metacognizione?

La metacognizione è il **pensare al pensiero**. Nel contesto degli agenti IA, significa costruire agenti che possono:

- **Autoriflettere** sui propri output e sul processo di ragionamento
- **Rilevare errori** e riprendersi agevolmente invece di fallire silenziosamente
- **Valutare** se le loro risposte sono complete e utili
- **Adattare** la loro strategia quando un approccio iniziale non funziona (ad esempio, ricorrendo a un sistema di backup)

Un agente metacognitivo non si limita a rispondere alle domande — monitora le proprie prestazioni e si regola in tempo reale.


## Strumenti Primari e di Backup

Un modello comune di metacognizione è la **strategia di fallback**. L'agente prova prima uno strumento primario; se fallisce (ad esempio, un errore 404), l'agente riconosce il fallimento e passa in modo trasparente a uno strumento di backup.

Questo rispecchia i sistemi del mondo reale in cui i servizi principali potrebbero non essere disponibili e gli agenti devono autodiagnosticare il problema prima di scegliere un percorso alternativo.

Di seguito definiamo due strumenti di ricerca voli:
- **Primario** — copre Parigi, Tokyo e Barcellona
- **Backup** — copre Berlino, Sydney e New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agente auto-riflettente con recupero dagli errori

L'agente qui sotto è istruito a provare prima il sistema di volo primario, riconoscere i guasti e passare in modo trasparente al sistema di backup. Dopo ogni risposta si auto-valuta brevemente per verificare se ha risposto completamente alla domanda dell'utente.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Schema di Autovalutazione

Un altro aspetto della metacognizione è la **autovalutazione**: un agente separato (o lo stesso agente in un secondo passaggio) rivede una risposta per completezza, accuratezza e utilità.

Di seguito creiamo un agente `ResponseEvaluator` che valuta le risposte degli agenti di viaggio su tre dimensioni.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sommario

In questa lezione hai imparato come costruire **agenti metacognitivi** usando il Microsoft Agent Framework:

- **Autoriflessione**: Agenti che monitorano il proprio ragionamento e comunicano in modo trasparente ciò che è accaduto.
- **Recupero dagli errori con soluzioni alternative**: Uno schema di strumento primario + di riserva in cui l'agente rileva i guasti (es. errori 404) e automaticamente prova una fonte alternativa.
- **Auto-valutazione**: Un agente valutatore separato che giudica le risposte per completezza, accuratezza e utilità.

Questi schemi rendono gli agenti più robusti, trasparenti e affidabili — qualità fondamentali per distribuzioni in produzione.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
